# 🛟 HTML Table Rescuer — Live Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Encephos/html-table-rescuer/blob/main/examples/demo.ipynb)

Extract complex HTML tables and convert them into clean **Markdown**, **JSON**, or **CSV** — built for LLM context windows and RAG pipelines.

The key feature: a **grid solver** that correctly resolves `rowspan` and `colspan`, so complex tables never end up as a misaligned mess.

Run the cells top to bottom (`Shift+Enter`) — no setup needed.

In [1]:
%pip install -q html-table-rescuer

Note: you may need to restart the kernel to use updated packages.


## The problem

This table uses `rowspan` and `colspan` — most converters shift every cell after the span and produce garbage. Watch the grid solver handle it:

In [2]:
from html_table_rescuer import TableParser

html = """
<table>
  <tr><th colspan="2">Department</th><th>Employee</th></tr>
  <tr><td rowspan="2">Engineering</td><td>Backend</td><td>Ada</td></tr>
  <tr><td>Frontend</td><td>Grace</td></tr>
  <tr><td colspan="2">Marketing</td><td>Margaret</td></tr>
</table>
"""

table = TableParser(html).parse()[0]
print(table.to_markdown())

| Department |  | Employee |
| --- | --- | --- |
| Engineering | Backend | Ada |
| dito (Engineering) | Frontend | Grace |
| Marketing |  | Margaret |


Every cell is exactly where it belongs — the `rowspan` cell is filled with `dito (…)` so an LLM reading row 3 still knows which department it's in.

And since the output is Markdown, notebooks render it as a real table:

In [3]:
from IPython.display import Markdown

Markdown(table.to_markdown())

| Department |  | Employee |
| --- | --- | --- |
| Engineering | Backend | Ada |
| dito (Engineering) | Frontend | Grace |
| Marketing |  | Margaret |

## Rowspan strategies

You decide how spanned cells are filled — `fill_dito` (default), `repeat`, or `empty`:

In [4]:
from html_table_rescuer import ParseConfig, RowspanStrategy

for strategy in RowspanStrategy:
    config = ParseConfig(rowspan_strategy=strategy)
    result = TableParser(html, config).parse()[0]
    print(f"--- {strategy.value} ---")
    print(result.to_markdown())
    print()

--- fill_dito ---
| Department |  | Employee |
| --- | --- | --- |
| Engineering | Backend | Ada |
| dito (Engineering) | Frontend | Grace |
| Marketing |  | Margaret |

--- repeat ---
| Department |  | Employee |
| --- | --- | --- |
| Engineering | Backend | Ada |
| Engineering | Frontend | Grace |
| Marketing |  | Margaret |

--- empty ---
| Department |  | Employee |
| --- | --- | --- |
| Engineering | Backend | Ada |
|  | Frontend | Grace |
| Marketing |  | Margaret |



## Structured exports: JSON and CSV

In [5]:
print(table.to_json())

[
  {
    "Department": "Engineering",
    "": "Backend",
    "Employee": "Ada"
  },
  {
    "Department": "dito (Engineering)",
    "": "Frontend",
    "Employee": "Grace"
  },
  {
    "Department": "Marketing",
    "": "",
    "Employee": "Margaret"
  }
]


In [6]:
print(table.to_csv())

Department,,Employee
Engineering,Backend,Ada
dito (Engineering),Frontend,Grace
Marketing,,Margaret



## Robust against broken real-world HTML

Invalid span values, HTML comments, unclosed tags — real-world HTML is messy. The parser survives all of it:

In [7]:
broken = """
<table>
  <tr><th>Product</th><th>Price</th>
  <tr><td colspan="abc">Widget<!-- internal note --></td><td>9.99
  <tr><td rowspan="0">Gadget</td><td>19.99</td>
</table>
"""

print(TableParser(broken).parse()[0].to_markdown())

| Product | Price |
| --- | --- |
| Widget | 9.99 |
| Gadget | 19.99 |


## Straight from the web

Wikipedia's [Help:Table](https://en.wikipedia.org/wiki/Help:Table) page is full of gnarly demo tables with spans — a perfect stress test:

In [8]:
import requests

resp = requests.get(
    "https://en.wikipedia.org/wiki/Help:Table",
    headers={"User-Agent": "html-table-rescuer-demo"},
    timeout=30,
)
tables = TableParser(resp.text).parse()
print(f"Found {len(tables)} tables. Here is one where the grid solver had work to do:\n")

# Pick the first table where a rowspan was actually resolved
spanned = next(t for t in tables if "dito" in t.to_markdown())
print(spanned.to_markdown())

Found 64 tables. Here is one where the grid solver had work to do:

| col1 | col2 | col3 | col4 |
| --- | --- | --- | --- |
| row1 | A |  | C |
| row2 | AA | BB | CC |
| row3 | AAA | BBB | CCC |
| row4 | AAAA | dito (BBB) | CCCC |


## There is also a CLI

`pip install` gives you an `html-table-rescuer` command — pipe HTML in, get Markdown/JSON/CSV out:

In [9]:
!echo '<table><tr><th>Region</th><th>Sales</th></tr><tr><td colspan="2">no data yet</td></tr></table>' | html-table-rescuer - --format json

[
  {
    "Region": "no data yet",
    "Sales": ""
  }
]


## RAG integrations

Every table becomes its own document, so retrieval never splits a table in half. Ready-made wrappers exist for all three major frameworks (each behind its own extra, e.g. `pip install "html-table-rescuer[langchain]"`):

```python
# LangChain
from html_table_rescuer.integrations.langchain import HTMLTableRescuerLoader
docs = HTMLTableRescuerLoader("page.html").load()

# LlamaIndex (works as file_extractor in SimpleDirectoryReader)
from html_table_rescuer.integrations.llamaindex import HTMLTableRescuerReader
docs = HTMLTableRescuerReader().load_data("page.html")

# Haystack (pipeline component, ParseConfig survives serialization)
from html_table_rescuer.integrations.haystack import HTMLTableRescuerConverter
docs = HTMLTableRescuerConverter().run(sources=["page.html"])["documents"]
```

## Links

- **GitHub:** [Encephos/html-table-rescuer](https://github.com/Encephos/html-table-rescuer)
- **PyPI:** [html-table-rescuer](https://pypi.org/project/html-table-rescuer/)
- **Blog:** [How to Stop LLMs from Hallucinating on Complex HTML Tables](https://dev.to/encephos/how-to-stop-llms-from-hallucinating-on-complex-html-tables-python-2e0k)